# 6. File I/O (Reading & Writing Data)

So far, all the data we have worked with has been typed directly into our code as arrays. In the real world, data lives in **files** — CSV spreadsheets, databases, or binary files. Before you can analyze anything, you need to know how to **read data in** and **write results out**.

This module covers NumPy's built-in tools for handling files, which is a critical step before working on any real data project.

We will cover:
1. **NumPy Binary Files**: The fastest way to save and load arrays using `np.save`, `np.load`, `.npy`, and `.npz`.
2. **Text / CSV Files**: Saving and loading human-readable data using `np.savetxt` and `np.loadtxt`.
3. **Loading Files with Missing Values**: Robustly loading messy real-world data using `np.genfromtxt`.
4. **Real-world Example**: A complete workflow — generate data, save to CSV, reload it, and analyze it.

In [1]:
import numpy as np
import os  # We'll use this to clean up files after each example

## 6.1 Saving & Loading NumPy Binary Files

The **fastest and most efficient** way to store NumPy arrays is in NumPy's own binary format (`.npy` and `.npz`). These files are not human-readable, but they preserve the array's **exact data type, shape, and values** perfectly — no rounding, no conversion.

- **`.npy`** — stores a **single** array.
- **`.npz`** — stores **multiple** arrays in one compressed file.

In [2]:
# --- np.save and np.load (single array) ---
measurements = np.array([1.5, 2.5, 3.5, 4.5, 5.5])

# Save to a .npy file (NumPy automatically adds the .npy extension)
np.save('measurements.npy', measurements)

# Load the array back from the file
loaded_measurements = np.load('measurements.npy')

print("Original array :", measurements)
print("Loaded array   :", loaded_measurements)
print("Are they equal?:", np.array_equal(measurements, loaded_measurements))
print("Dtype preserved:", loaded_measurements.dtype)

# Clean up
os.remove('measurements.npy')

Original array : [1.5 2.5 3.5 4.5 5.5]
Loaded array   : [1.5 2.5 3.5 4.5 5.5]
Are they equal?: True
Dtype preserved: float64


In [3]:
# --- np.savez and np.load (multiple arrays in one file) ---
# Great for saving a dataset split into features and labels together.
features = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0], [9.0, 10.0]])
labels   = np.array([0, 1, 0, 1, 0])

# Save both arrays into one .npz file using keyword arguments as names
np.savez('dataset.npz', features=features, labels=labels)

# Load the .npz file — it returns a dict-like object
loaded_dataset = np.load('dataset.npz')

print("Keys in .npz file:", list(loaded_dataset.keys()))
print("\nLoaded features:\n", loaded_dataset['features'])
print("\nLoaded labels:", loaded_dataset['labels'])

# Clean up
os.remove('dataset.npz')

Keys in .npz file: ['features', 'labels']

Loaded features:
 [[1.  2. ]
 [3.  4. ]
 [5.  6. ]
 [7.  8. ]
 [9.  10.]]

Loaded labels: [0 1 0 1 0]


## 6.2 Saving & Loading Text / CSV Files

Sometimes you need your data to be **human-readable**, or you need to share it with tools like Excel or pandas. For these cases, NumPy can read and write plain text and CSV files.

- **`np.savetxt`** — saves an array to a text file.
- **`np.loadtxt`** — loads data from a text file (all values must be present, no missing values).

In [4]:
# --- np.savetxt ---
# Each row in the array becomes a row in the file.
exam_results = np.array([
    [85, 92, 78],
    [70, 65, 88],
    [95, 98, 91],
    [60, 72, 55]
], dtype=float)

# Save as CSV with a header comment
np.savetxt(
    'exam_results.csv',
    exam_results,
    delimiter=',',          # Use comma as separator
    fmt='%.2f',             # Format each number to 2 decimal places
    header='Student exam results: Math, Science, English'
)

# Show the raw file content to understand the format
print("--- Raw file contents ---")
with open('exam_results.csv', 'r') as f:
    print(f.read())

# --- np.loadtxt ---
# Load the CSV back into a NumPy array
loaded_results = np.loadtxt(
    'exam_results.csv',
    delimiter=',',   # Same delimiter used when saving
    comments='#'     # Lines starting with '#' are treated as comments
)

print("--- Loaded array ---")
print(loaded_results)
print("\nShape:", loaded_results.shape)

# Clean up
os.remove('exam_results.csv')

--- Raw file contents ---
# Student exam results: Math, Science, English
85.00,92.00,78.00
70.00,65.00,88.00
95.00,98.00,91.00
60.00,72.00,55.00

--- Loaded array ---
[[85. 92. 78.]
 [70. 65. 88.]
 [95. 98. 91.]
 [60. 72. 55.]]

Shape: (4, 3)


## 6.3 Loading Files with Missing Values

Real-world data is often **messy** — some cells might be empty or have invalid values. `np.loadtxt` will crash if it encounters a missing value. That's where **`np.genfromtxt`** comes in: it can gracefully handle missing data by filling the gaps with `NaN`.

> **Tip**: When the data has missing values, `genfromtxt` automatically uses `float64` dtype because `NaN` is a float concept.

In [5]:
# Create a messy CSV file that simulates real-world missing data
messy_csv_content = """Math,Science,English
85,92,78
70,,88
95,98,
60,72,55"""

with open('messy_data.csv', 'w') as f:
    f.write(messy_csv_content)

print("--- Messy CSV file contents ---")
print(messy_csv_content)

# --- np.genfromtxt ---
# Loads the file and fills missing values with NaN automatically
messy_data = np.genfromtxt(
    'messy_data.csv',
    delimiter=',',
    skip_header=1,        # Skip the first row (column names)
    filling_values=np.nan # Fill missing cells with NaN
)

print("\n--- Loaded with np.genfromtxt ---")
print(messy_data)
print("\nMissing value positions:\n", np.isnan(messy_data))

# Use NaN-safe functions from Module 5 to analyze despite missing data
column_means = np.nanmean(messy_data, axis=0)
print("\nColumn means (ignoring NaN):", np.round(column_means, 2))

# Clean up
os.remove('messy_data.csv')

--- Messy CSV file contents ---
Math,Science,English
85,92,78
70,,88
95,98,
60,72,55

--- Loaded with np.genfromtxt ---
[[85. 92. 78.]
 [70. nan 88.]
 [95. 98. nan]
 [60. 72. 55.]]

Missing value positions:
 [[False False False]
 [False  True False]
 [False False  True]
 [False False False]]

Column means (ignoring NaN): [77.5  87.33 73.67]


## 6.4 Real-world Example: Save, Reload, and Analyze a Dataset

Let's put it all together. We'll simulate a realistic workflow:
1. **Generate** a dummy dataset (student scores for 5 subjects).
2. **Save** it to a CSV file.
3. **Reload** it from the file (as if someone gave us this file to analyze).
4. **Analyze** it using what we have learned so far.

In [6]:
# --- Step 1: Generate a dummy dataset ---
np.random.seed(42)  # For reproducibility

n_students = 20
n_subjects = 5
subjects   = ['Math', 'Science', 'English', 'History', 'Art']

# Generate random scores between 50 and 99
student_scores = np.random.randint(50, 100, size=(n_students, n_subjects))

print("=== Step 1: Generated Dataset (first 5 rows) ===")
print(student_scores[:5])

# --- Step 2: Save to CSV ---
np.savetxt(
    'student_scores.csv',
    student_scores,
    delimiter=',',
    fmt='%d',   # Integer format (no decimals)
    header=','.join(subjects)
)
print("\n=== Step 2: Saved to 'student_scores.csv' ===")

# --- Step 3: Reload from file ---
# Simulate receiving this CSV from someone else
reload_scores = np.loadtxt('student_scores.csv', delimiter=',', comments='#')

print("\n=== Step 3: Reloaded from file ===")
print(f"Shape: {reload_scores.shape}  |  Dtype: {reload_scores.dtype}")

# --- Step 4: Analyze ---
col_means = np.mean(reload_scores, axis=0)   # Mean per subject
col_stds  = np.std(reload_scores, axis=0)    # Std deviation per subject
col_min   = np.min(reload_scores, axis=0)    # Min score per subject
col_max   = np.max(reload_scores, axis=0)    # Max score per subject

print("\n=== Step 4: Analysis Results ===")
print(f"{'Subject':<16}" + "".join(f"{s:>8}" for s in subjects))
print(f"{'Mean Score':<16}" + "".join(f"{v:>8.1f}" for v in col_means))
print(f"{'Std Deviation':<16}" + "".join(f"{v:>8.1f}" for v in col_stds))
print(f"{'Min Score':<16}" + "".join(f"{v:>8.1f}" for v in col_min))
print(f"{'Max Score':<16}" + "".join(f"{v:>8.1f}" for v in col_max))

# Find the best-performing subject
best_subject_idx = np.argmax(col_means)
print(f"\nBest performing subject: {subjects[best_subject_idx]} (avg: {col_means[best_subject_idx]:.2f})")

# Count students with an average score >= 75
student_averages = np.mean(reload_scores, axis=1)  # Mean per student (row)
high_achievers   = np.sum(student_averages >= 75)
print(f"Total students with avg score >= 75: {high_achievers}")

# Clean up
os.remove('student_scores.csv')

=== Step 1: Generated Dataset (first 5 rows) ===
[[74 60 77 85 90]
 [62 88 54 73 66]
 [91 75 83 69 78]
 [55 67 92 81 70]
 [88 79 61 56 94]]

=== Step 2: Saved to 'student_scores.csv' ===

=== Step 3: Reloaded from file ===
Shape: (20, 5)  |  Dtype: float64

=== Step 4: Analysis Results ===
Subject         Math  Science  English  History   Art
Mean Score      72.8     73.1     72.9     73.3  74.9
Std Deviation   12.3     11.1     13.4     13.9  12.2
Min Score       50.0     55.0     50.0     50.0  52.0
Max Score       95.0     96.0     97.0     98.0  99.0

Best performing subject: Art (avg: 74.90)
Total students with avg score >= 75: 8
